# Gold - Export snapshot

Ultimo task del job de Gold (Fase 1 del roadmap): escribe un Parquet unico de
`weather.gold.training_dataset_v0` (solo `punto_prediccion = 'ana_74100000'`) mas un
`manifest.json` en el Volume, para que `notebooks_local/gold_export/export_gold_dataset.py`
lo baje sin necesitar un SQL warehouse encendido (Decision 016).

In [ ]:
import hashlib
import json
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F

GOLD_TABLE = 'weather.gold.training_dataset_v0'
PUNTO_PREDICCION = 'ana_74100000'

VOLUME_DIR = '/Volumes/weather/raw/gold_export_volume'
STAGING_DIR = f'{VOLUME_DIR}/_staging'
PARQUET_PATH = f'{VOLUME_DIR}/training_dataset_v0.parquet'
MANIFEST_PATH = f'{VOLUME_DIR}/manifest.json'

In [ ]:
df = spark.table(GOLD_TABLE).filter(F.col('punto_prediccion') == F.lit(PUNTO_PREDICCION))

bounds = df.agg(F.min('fecha').alias('min_fecha'), F.max('fecha').alias('max_fecha'), F.count('*').alias('rows')).first()
if bounds['rows'] == 0:
    raise ValueError(f'{GOLD_TABLE} no tiene filas para {PUNTO_PREDICCION}; no se exporta un snapshot vacio')

delta_version = DeltaTable.forName(spark, GOLD_TABLE).history(1).select('version').first()['version']
rows, fecha_min, fecha_max = bounds['rows'], bounds['min_fecha'], bounds['max_fecha']
print(f'Exportando version Delta {delta_version}: {rows} filas, {fecha_min} a {fecha_max}')

In [ ]:
# Un solo archivo de salida: coalesce(1) y despues renombramos el unico part-file al
# nombre fijo que espera el exportador local, en vez de dejar los nombres de Spark.
dbutils.fs.rm(STAGING_DIR, recurse=True)
df.coalesce(1).write.format('parquet').mode('overwrite').save(STAGING_DIR)

part_files = [f.path for f in dbutils.fs.ls(STAGING_DIR) if f.name.startswith('part-') and f.name.endswith('.parquet')]
if len(part_files) != 1:
    raise ValueError(f'Se esperaba 1 part-file en {STAGING_DIR}, se encontraron {len(part_files)}')

dbutils.fs.cp(part_files[0], PARQUET_PATH)
dbutils.fs.rm(STAGING_DIR, recurse=True)
print(f'Parquet escrito en {PARQUET_PATH}')

In [ ]:
# Los paths /Volumes/... son POSIX en el cluster: se pueden leer con I/O estandar de
# Python, sin pasar por Spark, para hashear el archivo tal cual queda en el Volume.
def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    'delta_version': delta_version,
    'rows': bounds['rows'],
    'fecha_min': str(bounds['min_fecha']),
    'fecha_max': str(bounds['max_fecha']),
    'punto_prediccion': PUNTO_PREDICCION,
    'columns': df.columns,
    'file_name': 'training_dataset_v0.parquet',
    'file_sha256': sha256_of(PARQUET_PATH),
    'exported_at': datetime.now(timezone.utc).isoformat(),
}

with open(MANIFEST_PATH, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(json.dumps(manifest, ensure_ascii=False, indent=2))